In [1]:
import pandas as pd
import sqlite3
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

# Combine datasets

In [2]:
gaming_data = pd.read_csv('../data/gaming_GAD_CLEAN.csv')
nhis_data = pd.read_csv('../data/nhis_GAD_CLEAN.csv')

In [3]:
# Check what columns are different between the two dataframes

set(gaming_data.columns) ^ set(nhis_data.columns)

{'Degree',
 'Gender',
 'game',
 'hours',
 'participant_id_OLD',
 'platform',
 'playstyle',
 'reason',
 'region_id'}

In [4]:
df = pd.concat([gaming_data, nhis_data], ignore_index=True, sort=False)
df.head()

# Reference: https://pandas.pydata.org/docs/user_guide/merging.html

,Unnamed: 0,participant_id,year,GAD1,GAD2,GAD3,GAD4,GAD5,GAD6,GAD7,GAD_total,GAD_cat,sex_id,Degree,age,survey_id,education_id,residence,game,platform,hours,reason,playstyle,Gender,region_id,participant_id_OLD
0,0,1,2015,0,0,0,0,1,0,0,1,1,1,Bachelor (or equivalent),25,1,8,USA,Skyrim,"Console (PS, Xbox, ...)",15.0,having fun,Singleplayer,Male,NaN,NaN
1,1,2,2015,1,2,2,2,0,1,0,8,2,1,Bachelor (or equivalent),41,1,8,USA,Other,PC,8.0,having fun,Multiplayer - online - with strangers,Male,NaN,NaN
2,2,3,2015,0,2,2,0,0,3,1,8,2,2,Bachelor (or equivalent),32,1,8,DEU,Other,PC,0.0,having fun,Singleplayer,Female,NaN,NaN
3,3,4,2015,0,0,0,0,0,0,0,0,1,1,Bachelor (or equivalent),28,1,8,USA,Other,PC,20.0,improving,Multiplayer - online - with online acquaintanc...,Male,NaN,NaN
4,4,5,2015,2,1,2,2,2,3,2,14,3,1,High school diploma (or equivalent),19,1,12,KOR,Other,"Console (PS, Xbox, ...)",20.0,having fun,Multiplayer - online - with strangers,Male,NaN,NaN


In [5]:
# This remains from an unknown issue before where both datasets had a participant with this id
filtered_data = df.loc[df['participant_id'] == 43]
filtered_data

,Unnamed: 0,participant_id,year,GAD1,GAD2,GAD3,GAD4,GAD5,GAD6,GAD7,GAD_total,GAD_cat,sex_id,Degree,age,survey_id,education_id,residence,game,platform,hours,reason,playstyle,Gender,region_id,participant_id_OLD
42,42,43,2015,0,0,0,0,0,0,0,0,1,1,Bachelor (or equivalent),27,1,8,USA,Other,PC,12.0,having fun,Multiplayer - online - with strangers,Male,NaN,NaN


In [6]:
# Confirming that the data fix was successful

filtered_data = df.loc[df['GAD_total'] > 21]
filtered_data

# GAD_total needs to be recalculated after data correction

,Unnamed: 0,participant_id,year,GAD1,GAD2,GAD3,GAD4,GAD5,GAD6,GAD7,GAD_total,GAD_cat,sex_id,Degree,age,survey_id,education_id,residence,game,platform,hours,reason,playstyle,Gender,region_id,participant_id_OLD


In [7]:
# Confirm that no number is >21
df['GAD_total'].unique()

array([ 1,  8,  0, 14, 12, 10, 19,  3,  2,  4, 15,  5,  6,  7, 13, 11,  9,
       18, 16, 21, 17, 20])

# Build Relational Tables
## Create unique dataframes

In [17]:
# ================
# Fact tables
# ================

participant_df = df[['participant_id', 'survey_id', 'sex_id', 'education_id', 'residence', 'age', 'region_id']].drop_duplicates()

GAD_response_df = df[['participant_id', 'GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'GAD_total', 'GAD_cat']]

gaming_response_df = df[['participant_id', 'survey_id', 'game', 'platform', 'hours', 'reason', 'playstyle']].drop_duplicates()

# ================
# Lookup tables
# ================

survey_df = pd.DataFrame({
    'survey_id': [1, 2],
    'survey_name': ['Gamers', 'NHIS'],
    'year': [2015, 2019],
    'population':['global', 'America']
})


GAD_categories = {
    1:'None/Minimal', 
    2:'Mild', 
    3:'Moderate', 
    4:'Severe', 
    8:'Not Ascertained'
}

GAD_cat_df = (
    pd.Series(GAD_categories, name='cat_name')
        .rename_axis('GAD_cat')
        .reset_index()
)

sex_lookup = {
    1: 'Male', 
    2:'Female', 
    7:'Refused', 
    8:'Not Ascertained',
    9:"Don't Know"
}
sex_df = (
    pd.Series(sex_lookup, name='sex_name')
        .rename_axis('sex_id')
        .reset_index()
)


degree_lookup = {
    0:'Never attended/kindergarten only', 
    1: 'Grade 1-11',
    2: '12th grade, no diploma',
    3: 'GED or equivalent',
    4: 'High School Graduate',
    5: 'Some college, no degree',
    6: 'Associate degree: occupational, technical, or vocational program',
    7: 'Associate degree: academic program',
    8: "Bachelor's degree (Example: BA, AB, BS, BBA)",
    9: "Master's degree or equivalent (Example: MA, MS, MEng, MEd, MBA)",
    10: "Professional School degree (Example: MD, DDS, DVM, JD)",
    11: "Doctoral degree (Example: PhD, EdD)",
    12: "High school diploma (or equivalent)",
    13: "Ph.D., Psy. D., MD (or equivalent)",
    97: "Refused",
    98: "Not Ascertained",
    99:"Don't Know"
}
education_df = (
    pd.Series(degree_lookup, name='level')
        .rename_axis('education_id')
        .reset_index()
)


region_lookup = {
    1: 'Northeast',
    2: 'Midwest',
    3: 'South', 
    4: 'West'
}
region_df = (
    pd.Series(region_lookup, name='name')
        .rename_axis('region_id')
        .reset_index()
)

# I conferred with ChatGPT to decide how to set up the lookup tables
# it recommended using dictionaries to set them up to make it easier to 
# read and maintain. I did this for the longer descriptions

## Add custom id columns to GAD_response_df and gaming_response_df
This will be a unique identifier and primary key

Reference: https://www.geeksforgeeks.org/python/create-a-pandas-column-using-for-loop/

In [16]:
def add_custom_id_column(dataframe, new_col_name, first_id_no):
    if new_col_name in dataframe:
        return
    else:
        new_col_data = []
        id_no = first_id_no

        for participant in dataframe['participant_id']:
            new_col_data.append(id_no)
            id_no += 1

        dataframe[new_col_name] = new_col_data

        col_to_move = dataframe.pop(new_col_name)
        dataframe.insert(0, new_col_name, col_to_move)

add_custom_id_column(GAD_response_df, 'GAD_response_id', 1000)

GAD_response_df.head()

,GAD_response_id,participant_id,GAD1,GAD2,GAD3,GAD4,GAD5,GAD6,GAD7,GAD_total,GAD_cat
0,1000,1,0,0,0,0,1,0,0,1,1
1,1001,2,1,2,2,2,0,1,0,8,2
2,1002,3,0,2,2,0,0,3,1,8,2
3,1003,4,0,0,0,0,0,0,0,0,1
4,1004,5,2,1,2,2,2,3,2,14,3


In [18]:
add_custom_id_column(gaming_response_df, 'gaming_response_id', 2000)

gaming_response_df.head()

,gaming_response_id,participant_id,survey_id,game,platform,hours,reason,playstyle
0,2000,1,1,Skyrim,"Console (PS, Xbox, ...)",15.0,having fun,Singleplayer
1,2001,2,1,Other,PC,8.0,having fun,Multiplayer - online - with strangers
2,2002,3,1,Other,PC,0.0,having fun,Singleplayer
3,2003,4,1,Other,PC,20.0,improving,Multiplayer - online - with online acquaintanc...
4,2004,5,1,Other,"Console (PS, Xbox, ...)",20.0,having fun,Multiplayer - online - with strangers


## Create database tables

In [19]:
conn = sqlite3.connect('../data/mentalhealth.db')

def df_to_table(input_dataframe, output_table_name):
    input_dataframe.to_sql(output_table_name, conn, index=False, if_exists='replace')

df_to_table(participant_df, 'participant')
df_to_table(GAD_response_df, 'GAD_response')
df_to_table(gaming_response_df, 'gaming_response')
df_to_table(survey_df, 'survey')
df_to_table(GAD_cat_df, 'GAD_category')
df_to_table(sex_df, 'sex')
df_to_table(education_df, 'education')
df_to_table(region_df, 'region')

# Exploratory Data Analysis

## QUERIES

### Participant data and scores (all_data_scores_r)

In [ ]:
all_data_scores_q = """
SELECT
    p.participant_id,
    p.survey_id,
    p.sex_id,
    p.education_id,
    p.residence,
    p.age,
    p.region_id,
    r.GAD1,
    r.GAD2,
    r.GAD3,
    r.GAD4,
    r.GAD5,
    r.GAD6,
    r.GAD7,
    r.GAD_total,
    r.GAD_cat
FROM participants p
JOIN responses r on r.participant_id = p.participant_id
"""

all_data_scores_r = pd.read_sql(all_data_scores_q, conn)
all_data_scores_r

### Participant data and scores (USA) (usa_data_scores_r)

In [ ]:
usa_data_scores_q = """
SELECT
    p.participant_id,
    p.survey_id,
    p.sex_id,
    p.education_id,
    p.residence,
    p.age,
    p.region_id,
    r.GAD1,
    r.GAD2,
    r.GAD3,
    r.GAD4,
    r.GAD5,
    r.GAD6,
    r.GAD7,
    r.GAD_total,
    r.GAD_cat
FROM participants p
JOIN responses r on r.participant_id = p.participant_id
WHERE p.residence = "USA"
"""

usa_data_scores_r = pd.read_sql(usa_data_scores_q, conn)
usa_data_scores_r

### NHIS data and scores (nhis_data_scores_r)

In [ ]:
nhis_data_scores_q = """
SELECT
    p.participant_id,
    p.survey_id,
    p.sex_id,
    p.education_id,
    p.residence,
    p.age,
    p.region_id,
    r.GAD1,
    r.GAD2,
    r.GAD3,
    r.GAD4,
    r.GAD5,
    r.GAD6,
    r.GAD7,
    r.GAD_total,
    r.GAD_cat
FROM participants p
JOIN responses r on r.participant_id = p.participant_id
WHERE p.survey_id = 2
"""

nhis_data_scores_r = pd.read_sql(nhis_data_scores_q, conn)
nhis_data_scores_r

### Gamer data and scores (USA) (usa_gamer_data_scores_r)

In [ ]:
usa_gamer_data_scores_q = """
SELECT
    p.participant_id,
    p.survey_id,
    p.sex_id,
    p.education_id,
    p.residence,
    p.age,
    p.region_id,
    r.GAD1,
    r.GAD2,
    r.GAD3,
    r.GAD4,
    r.GAD5,
    r.GAD6,
    r.GAD7,
    r.GAD_total,
    r.GAD_cat
FROM participants p
JOIN responses r on r.participant_id = p.participant_id
WHERE p.survey_id = 1
    AND p.residence = "USA"
"""

usa_gamer_data_scores_r = pd.read_sql(usa_gamer_data_scores_q, conn)
usa_gamer_data_scores_r

### Gamer data and scores (World) (all_gamer_data_scores_r)

In [ ]:
all_gamer_data_scores_q = """
SELECT
    p.participant_id,
    p.survey_id,
    p.sex_id,
    p.education_id,
    p.residence,
    p.age,
    p.region_id,
    r.GAD1,
    r.GAD2,
    r.GAD3,
    r.GAD4,
    r.GAD5,
    r.GAD6,
    r.GAD7,
    r.GAD_total,
    r.GAD_cat
FROM participants p
JOIN responses r on r.participant_id = p.participant_id
WHERE p.survey_id = 1
"""

all_gamer_data_scores_r = pd.read_sql(all_gamer_data_scores_q, conn)
all_gamer_data_scores_r

### Scores by sex (sex_data_scores_r)

In [ ]:
sex_data_scores_q = """
SELECT
    p.sex_id,
    s.sex_name,
    AVG(r.GAD1),
    AVG(r.GAD2),
    AVG(r.GAD3),
    AVG(r.GAD4),
    AVG(r.GAD5),
    AVG(r.GAD6),
    AVG(r.GAD7),
    AVG(r.GAD_total),
    AVG(r.GAD_cat),
    count(p.sex_id) as total_participants
FROM participants p
JOIN responses r on r.participant_id = p.participant_id
JOIN sex s on s.sex_id = p.sex_id
GROUP BY p.sex_id
"""

sex_data_scores_r = pd.read_sql(sex_data_scores_q, conn)
sex_data_scores_r

### Scores by age (usa_age_data_scores_r)
Note that this *may* skew results somewhat as the majority of participants of the gamer survey were in their early 20s.

In [ ]:
usa_age_data_scores_q = """
SELECT
    p.age,
    AVG(r.GAD1),
    AVG(r.GAD2),
    AVG(r.GAD3),
    AVG(r.GAD4),
    AVG(r.GAD5),
    AVG(r.GAD6),
    AVG(r.GAD7),
    AVG(r.GAD_total),
    AVG(r.GAD_cat),
    count(p.age) as total_participants
FROM participants p
JOIN responses r on r.participant_id = p.participant_id
GROUP BY p.age
"""

usa_age_data_scores_r = pd.read_sql(usa_age_data_scores_q, conn)
usa_age_data_scores_r

### Scores by education (all_gamer_data_scores_r)

In [ ]:
all_gamer_data_scores_q = """
SELECT
    p.education_id,
    AVG(r.GAD1),
    AVG(r.GAD2),
    AVG(r.GAD3),
    AVG(r.GAD4),
    AVG(r.GAD5),
    AVG(r.GAD6),
    AVG(r.GAD7),
    AVG(r.GAD_total),
    AVG(r.GAD_cat),
    count(p.education_id) as total_participants
FROM participants p
JOIN responses r on r.participant_id = p.participant_id
GROUP BY p.education_id
"""

all_gamer_data_scores_r = pd.read_sql(all_gamer_data_scores_q, conn)
all_gamer_data_scores_r

## Visualizations

- box & whisker plot of scores (total and by survey)

### Heat map: Is there an observable and unexptected correlation in the data?

In [ ]:
# def visualize_heat_map():


numeric_usa_df = usa_data_scores_r.select_dtypes(include=[np.number])
corr = numeric_usa_df.corr()

im = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
plt.colorbar(im, label='Pearson Correllation')

plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)

plt.show()

There do not appear to be any strong, unexpected correlations revealed by this heat map.

### Pie chart: What is the division of participants' identified sex?

In [ ]:
category = 'sex_name'
values = 'total_participants'

sex_totals = sex_data_scores_r.groupby(category)[values].sum()
sex_totals

In [ ]:
plt.figure(figsize=(8,8))
plt.pie(
    sex_totals,
    labels=None,
    autopct='%1.2f%%'
)

plt.title('Greater Participation by Males')

plt.legend(
    labels=sex_totals.index,
    loc='lower right',
    title='Sex'
)

plt.tight_layout()
plt.show()

### Bar chart: What is the average GAD score for each survey? (USA)

In [ ]:
usa_data_scores_r

In [ ]:
avg_score = usa_data_scores_r.groupby('survey_id')['GAD_total'].mean()
avg_score

In [ ]:
surveys = ('Gamers', 'NHIS')

fig, ax = plt.subplots()
bottom = np.zeros(2)

for survey in avg_score.items():
    p = ax.bar(surveys, avg_score, label=survey, bottom=bottom)
    ax.bar_label(p, label_type='center')

ax.set_title('Higher Average Anxiety Levels in Gamers (USA)')

plt.show()

# Reference: 
#   x-tick labels: https://matplotlib.org/stable/gallery/lines_bars_and_markers/bar_label_demo.html

### Count plots of avg scores
  - totals
  - by survey
  - (total and by survey)
    - by sex
    - by age
    - by education

#### How do the GAD score totals compare across all participants?

In [ ]:
usa_data_scores_r

In [ ]:
usa_score_count = usa_data_scores_r['GAD_total'].value_counts()

plt.bar(usa_score_count.index, usa_score_count.values)

plt.show()

In [ ]:
# investigating  why any number is above total possible GAD score of 21
# usa_data_scores_r['GAD_total'].unique()

In [ ]:
# filtered_scores = usa_data_scores_r.loc[usa_data_scores_r['GAD_total'] > 21]
# filtered_scores

# The highest number that SHOULD be possible in the GAD1-7 columns is 3.

#### How do the GAD score totals compare across USA gamer participants?

In [ ]:
usa_gamer_data_scores_r = usa_gamer_data_scores_r['GAD_total'].value_counts()

plt.bar(usa_gamer_data_scores_r.index, usa_gamer_data_scores_r.values)

plt.show()

#### How do the GAD score totals compare across NHIS participants?

In [ ]:
nhis_data_scores_r = nhis_data_scores_r['GAD_total'].value_counts()

plt.bar(nhis_data_scores_r.index, nhis_data_scores_r.values)

plt.show()

In [ ]:
# Confirming numbers are different between usa, gaming, and nhis
usa_data_scores_r['GAD_total'].value_counts()

In [ ]:
# usa_gamer_data_scores_r['GAD_total'].value_counts()
# usa_gamer_data_scores_r.info()
usa_gamer_data_scores_r

In [ ]:
nhis_data_scores_r